# Risk Parity Portfolio Model - Interactive Prototype

A modular implementation for risk-balanced portfolio construction and monitoring.

**Key Features:**
- Compute long-only Risk Parity weights from covariance matrices
- Calculate each asset's contribution to total portfolio risk  
- Run rolling backtests to see how RP weights evolve over time
- Detect significant shifts in portfolio composition
- Visualize weights, risk contributions, and regime shifts

**What is Risk Parity?**

Risk Parity is a portfolio construction approach that allocates based on **equal risk contribution** rather than equal capital weights. Each asset should contribute equally to total portfolio risk, regardless of its volatility or expected return.

This notebook lets you interactively explore Risk Parity on multi-asset portfolios.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from datetime import datetime, timedelta
import warnings
#Test

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2f}')

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

warnings.filterwarnings('ignore')

## 2. Covariance Estimation Functions

The foundation of Risk Parity is the covariance matrix. Here we implement a simple rolling sample covariance estimator, but the structure allows easy swapping with EWMA or shrinkage estimators.

In [ ]:
def estimate_rolling_covariance(returns, window):
    """
    Estimate rolling sample covariance matrix from returns.
    
    Parameters:
        returns (pd.DataFrame): Shape (T, N) - T time periods, N assets
        window (int): Rolling window size
    
    Returns:
        dict: 'dates' (DatetimeIndex), 'cov_matrices' (list), 'cov_dict' (dict for lookup)
    """
    if window > len(returns):
        raise ValueError(f"Window ({window}) > data length ({len(returns)})")
    
    dates = []
    cov_matrices = []
    
    for i in range(window, len(returns) + 1):
        window_returns = returns.iloc[i - window:i].values
        cov_matrix = np.cov(window_returns.T)
        dates.append(returns.index[i - 1])
        cov_matrices.append(cov_matrix)
    
    cov_dict = {date: cov for date, cov in zip(dates, cov_matrices)}
    
    return {
        'dates': pd.DatetimeIndex(dates),
        'cov_matrices': cov_matrices,
        'cov_dict': cov_dict
    }

print("✓ Covariance estimation function defined")

## 3. Risk Parity Weight Optimization

The core of the model: find weights so each asset contributes equally to total portfolio risk.

**Objective:** Minimize the sum of squared deviations between each asset's risk contribution and the target (equal) contribution.

**Constraints:** 
- Weights sum to 1
- Weights ≥ 0 (long-only)

In [ ]:
def compute_rp_weights(cov_matrix, long_only=True):
    """
    Compute long-only Risk Parity weights from a covariance matrix.
    
    Parameters:
        cov_matrix (np.ndarray): Covariance matrix (N, N)
        long_only (bool): If True, enforce non-negative weights
    
    Returns:
        weights (np.ndarray): RP weights summing to 1
        success (bool): Whether optimization converged
    """
    n = cov_matrix.shape[0]
    
    def objective(w):
        """Minimize sum of squared deviations in risk contribution."""
        portfolio_vol = np.sqrt(w @ cov_matrix @ w)
        if portfolio_vol < 1e-10:
            return 1e10

        mrc = cov_matrix @ w
        rc = w * mrc / portfolio_vol
        target_rc = portfolio_vol / n

        return np.sum((rc - target_rc) ** 2)
    
    # Initial guess: inverse volatility
    diag_cov = np.sqrt(np.diag(cov_matrix))
    w0 = 1.0 / diag_cov
    w0 = w0 / np.sum(w0)
    
    # Constraints: sum to 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    
    # Bounds: non-negative and at most 100%
    bounds = [(0, 1) for _ in range(n)] if long_only else [(None, None) for _ in range(n)]
    
    # Optimize
    result = minimize(
        objective,
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-9}
    )
    
    return result.x, result.success

print("✓ Risk Parity weight optimization function defined")

In [ ]:
#Modificeret
def compute_rp_weights(cov_matrix, long_only=True):
    """
    Compute long-only Risk Parity weights from a covariance matrix.

    Parameters:
        cov_matrix (np.ndarray): Covariance matrix (N, N)
        long_only (bool): If True, enforce non-negative weights

    Returns:
        weights (np.ndarray): RP weights summing to 1
        success (bool): Whether optimization converged
    """
    n = cov_matrix.shape[0]

    def objective(w):
        portfolio_var = w @ cov_matrix @ w
        if portfolio_var < 1e-10:
            return 1e10
        mrc = cov_matrix @ w
        rc = w * mrc / portfolio_var  # relative RC, summer til 1
        target = 1.0 / n
        return np.sum((rc - target) ** 2)

    # Initial guess: inverse volatility
    diag_cov = np.sqrt(np.diag(cov_matrix))
    w0 = 1.0 / diag_cov
    w0 = w0 / np.sum(w0)

    # Constraints: sum to 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]

    # Bounds: non-negative and at most 100%
    bounds = [(0, 1) for _ in range(n)] if long_only else [(None, None) for _ in range(n)]

    # Optimize
    result = minimize(
        objective,
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-9}
    )

    return result.x, result.success

print("✓ Risk Parity weight optimization function defined")


In [ ]:
def compute_risk_contributions(weights, cov_matrix):
    portfolio_vol = np.sqrt(weights @ cov_matrix @ weights)
    portfolio_var = portfolio_vol ** 2
    
    if portfolio_vol < 1e-10:
        return np.zeros_like(weights), portfolio_vol
    
    mrc = cov_matrix @ weights
    risk_contrib = weights * mrc / portfolio_var
    
    return risk_contrib, portfolio_vol

def compute_portfolio_volatility(weights, cov_matrix):
    """Compute portfolio volatility (standard deviation)."""
    variance = weights @ cov_matrix @ weights
    return np.sqrt(max(variance, 0))

print("✓ Risk contribution functions defined")

## 5. Rolling Backtest Engine

Main function: iterates through history, estimates covariance at each rebalance date, computes RP weights, and tracks evolution.

In [ ]:
def rolling_rp_backtest(returns, rebalance_freq='M', lookback_window=252):
    """
    Run a rolling Risk Parity backtest on historical returns.
    
    Parameters:
        returns (pd.DataFrame): Shape (T, N) - historical returns
        rebalance_freq (str): 'D', 'W', 'M', 'Q', 'Y'
        lookback_window (int): Rolling window size for covariance
    
    Returns:
        weights_df, risk_contrib_df, portfolio_vol_df, rebalance_dates
    """
    if returns.empty or len(returns) < lookback_window:
        raise ValueError(f"Returns must have at least {lookback_window} observations")
    
    # Generate candidate rebalance dates
    candidate_dates = pd.date_range(
        start=returns.index[lookback_window - 1],
        end=returns.index[-1],
        freq=rebalance_freq
    )
    
    # Find actual rebalance dates (dates that exist in returns)
    rebalance_dates = [d for d in candidate_dates if d in returns.index]
    
    if not rebalance_dates:
        raise ValueError(f"No valid rebalance dates found for frequency '{rebalance_freq}'")
    
    weights_list = []
    risk_contrib_list = []
    portfolio_vol_list = []
    asset_names = returns.columns.tolist()
    
    # Loop through rebalance dates
    for rebal_date in rebalance_dates:
        data_idx = returns.index.get_loc(rebal_date)
        start_idx = max(0, data_idx - lookback_window + 1)
        window_returns = returns.iloc[start_idx:data_idx + 1]
        
        # Estimate covariance
        cov_matrix = np.cov(window_returns.T)*252
        
        # Compute RP weights
        weights, success = compute_rp_weights(cov_matrix, long_only=True)
        
        if not success:
            warnings.warn(f"RP optimization did not converge at {rebal_date}")
            diag_cov = np.sqrt(np.diag(cov_matrix))
            weights = 1.0 / diag_cov
            weights = weights / np.sum(weights)
        
        # Compute risk contributions
        risk_contrib, portfolio_vol = compute_risk_contributions(weights, cov_matrix)
        
        weights_list.append(weights)
        risk_contrib_list.append(risk_contrib)
        portfolio_vol_list.append(portfolio_vol)
    
    # Build DataFrames
    weights_df = pd.DataFrame(
        weights_list,
        index=pd.DatetimeIndex(rebalance_dates),
        columns=asset_names
    )
    
    risk_contrib_df = pd.DataFrame(
        risk_contrib_list,
        index=pd.DatetimeIndex(rebalance_dates),
        columns=asset_names
    )
    
    portfolio_vol_df = pd.DataFrame(
        portfolio_vol_list,
        index=pd.DatetimeIndex(rebalance_dates),
        columns=['Portfolio Vol']
    )
    
    return weights_df, risk_contrib_df, portfolio_vol_df, pd.DatetimeIndex(rebalance_dates)

print("✓ Rolling backtest engine defined")

## 6. Regime Shift Indicator

Detect significant changes in Risk Parity weights, which can indicate structural changes in cross-asset relationships.

In [ ]:
def detect_weight_regime_shifts(weights_df, lookback=20, threshold=0.05):
    """
    Detect significant changes in Risk Parity weights over time.
    
    Parameters:
        weights_df (pd.DataFrame): RP weights from backtest
        lookback (int): Number of periods for rolling mean
        threshold (float): Threshold for flagging regime shift
    
    Returns:
        regime_shift_df, weight_changes_df
    """
    weight_changes = weights_df.diff().abs()
    total_change = weight_changes.sum(axis=1)
    regime_shift = (total_change > threshold).astype(int)
    
    regime_shift_df = pd.DataFrame(regime_shift, columns=['regime_shift'])
    
    return regime_shift_df, weight_changes

print("✓ Regime shift detection function defined")

## 7. Load and Prepare Data

Generate or load synthetic multi-asset returns. For this interactive notebook, we'll create realistic synthetic data with correlations.

In [ ]:
def generate_synthetic_returns(n_assets=5, n_days=1000, seed=42):
    """Generate realistic synthetic multi-asset returns."""
    np.random.seed(seed)
    
    asset_names = ['Equities', 'Bonds', 'Commodities', 'Real Estate', 'Credit'][:n_assets]
    
    # Set up realistic correlation structure
    if n_assets == 5:
        correlation_matrix = np.array([
            [1.00, -0.20, 0.10, 0.80, 0.70],
            [-0.20, 1.00, -0.10, 0.10, -0.30],
            [0.10, -0.10, 1.00, 0.20, 0.30],
            [0.80, 0.10, 0.20, 1.00, 0.50],
            [0.70, -0.30, 0.30, 0.50, 1.00],
        ])
        volatilities = np.array([0.15, 0.05, 0.20, 0.12, 0.08])
    else:
        correlation_matrix = np.eye(n_assets)
        volatilities = np.ones(n_assets) * 0.10
    
    D = np.diag(volatilities)
    covariance_matrix = D @ correlation_matrix @ D
    
    # Generate daily returns
    daily_returns = np.random.multivariate_normal(
        mean=np.zeros(n_assets),
        cov=covariance_matrix / 252,
        size=n_days
    )
    
    dates = pd.date_range(start='2020-01-01', periods=n_days, freq='B')
    returns_df = pd.DataFrame(daily_returns, index=dates, columns=asset_names)
    
    return returns_df

# Generate synthetic returns
print("Generating synthetic multi-asset returns...")
returns = generate_synthetic_returns(n_assets=5, n_days=1000)
print(f"✓ Generated {len(returns)} daily returns for {returns.shape[1]} assets")
print(f"  Period: {returns.index[0].strftime('%Y-%m-%d')} to {returns.index[-1].strftime('%Y-%m-%d')}")
print(f"\nFirst 5 rows of returns:")
print(returns.head())

## 8. Run Rolling Backtest

Execute the backtest with monthly rebalancing and a 1-year rolling window.

In [ ]:
print("\nRunning rolling Risk Parity backtest...")
print("  Rebalancing frequency: Monthly")
print("  Lookback window: 252 days (1 year)")
print("  Optimization: Long-only, equal risk contribution")
print()

weights_df, risk_contrib_df, portfolio_vol_df, rebalance_dates = rolling_rp_backtest(
    returns=returns,
    rebalance_freq='ME',  # Monthly (end of month)
    lookback_window=252
)

print(f"✓ Backtest completed: {len(weights_df)} rebalances")
print(f"  Date range: {weights_df.index[0].strftime('%Y-%m-%d')} to {weights_df.index[-1].strftime('%Y-%m-%d')}")

# Detect regime shifts
regime_shift_df, weight_changes_df = detect_weight_regime_shifts(weights_df, lookback=20, threshold=0.05)

print(f"\nLatest Risk Parity Weights ({weights_df.index[-1].strftime('%Y-%m-%d')}):")
print(weights_df.iloc[-1])

## 9. Visualize Risk Parity Weights Over Time

Track how portfolio allocations evolve. Each color represents one asset class.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

weights_df.plot(kind='area', stacked=True, ax=ax, alpha=0.75)

ax.set_title('Risk Parity Portfolio Weights Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Weight')
ax.set_ylim([0, 1])
ax.legend(title='Assets', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Weights visualization created")

## 10. Visualize Risk Contributions

Compare portfolio weights (left) with risk contributions (right) for the latest date. Risk Parity aims for equal risk contributions despite different weights.

In [ ]:
latest_date = weights_df.index[-1]
latest_weights = weights_df.iloc[-1]
latest_contrib = risk_contrib_df.iloc[-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Risk Contributions
latest_contrib.plot(kind='bar', ax=ax1, color='steelblue', alpha=0.8)
ax1.set_title(f'Risk Contributions ({latest_date.strftime("%Y-%m-%d")})', 
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Risk Contribution to Total Vol')
ax1.set_xlabel('')
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=latest_contrib.mean(), color='red', linestyle='--', 
            label=f'Mean: {latest_contrib.mean():.4f}', linewidth=2)
ax1.legend()
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Weights
latest_weights.plot(kind='bar', ax=ax2, color='coral', alpha=0.8)
ax2.set_title(f'Portfolio Weights ({latest_date.strftime("%Y-%m-%d")})', 
              fontsize=12, fontweight='bold')
ax2.set_ylabel('Portfolio Weight (%)')
ax2.set_xlabel('')
ax2.grid(True, alpha=0.3, axis='y')
[ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))]
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("✓ Risk contributions visualization created")
print(f"\nRisk Contribution Summary (as % of total portfolio risk):")
rc_pct = (latest_contrib / latest_contrib.sum()) * 100
for asset, contrib_pct in rc_pct.items():
    print(f"  {asset:15s}: {contrib_pct:5.1f}%")

## 11. Visualize Regime Shift Indicator Over Time

Track when the portfolio allocation changes significantly. Regime shifts may indicate changes in cross-asset correlations or volatility structure.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Regime shift indicator
ax1.fill_between(regime_shift_df.index, 0, regime_shift_df['regime_shift'],
                 alpha=0.3, color='red', label='Regime Shift Detected')
ax1.plot(regime_shift_df.index, regime_shift_df['regime_shift'],
         color='red', linewidth=2, drawstyle='steps-post')
ax1.set_ylabel('Regime Shift Indicator', fontweight='bold')
ax1.set_title('Risk Parity Regime Shift Detection (Top) and Weight Evolution (Bottom)', 
              fontsize=13, fontweight='bold')
ax1.set_ylim([-0.1, 1.1])
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')

# Weights over time
weights_df.plot(ax=ax2, linewidth=2, alpha=0.8)
ax2.set_ylabel('Portfolio Weight', fontweight='bold')
ax2.set_xlabel('Date', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(title='Assets', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.set_ylim([0, max(weights_df.max()) * 1.05])

plt.tight_layout()
plt.show()

print("✓ Regime shift visualization created")
num_shifts = regime_shift_df['regime_shift'].sum()
print(f"\nRegime Shift Summary:")
print(f"  Total shifts detected: {int(num_shifts)} out of {len(regime_shift_df)} rebalances")
print(f"  Shift frequency: {num_shifts / len(regime_shift_df) * 100:.1f}%")

## 12. Summary Statistics & Analysis

In [ ]:
print("=" * 70)
print("RISK PARITY BACKTEST SUMMARY")
print("=" * 70)

print(f"\nBacktest Period: {weights_df.index[0].strftime('%Y-%m-%d')} to {weights_df.index[-1].strftime('%Y-%m-%d')}")
print(f"Number of Rebalances: {len(weights_df)}")
print(f"Number of Assets: {len(weights_df.columns)}")

print("\n--- Latest Weights ---")
for asset, weight in latest_weights.items():
    print(f"  {asset:15s}: {weight:7.2%}")

print("\n--- Portfolio Volatility Statistics ---")
vol_series = portfolio_vol_df['Portfolio Vol']
print(f"  Mean:              {vol_series.mean():7.4f}")
print(f"  Min:               {vol_series.min():7.4f}")
print(f"  Max:               {vol_series.max():7.4f}")
print(f"  Std Dev:           {vol_series.std():7.4f}")

print("\n--- Average Weight by Asset ---")
for asset, mean_weight in weights_df.mean().items():
    print(f"  {asset:15s}: {mean_weight:7.2%}")

print("\n--- Weight Range (Min - Max) by Asset ---")
for asset in weights_df.columns:
    min_w = weights_df[asset].min()
    max_w = weights_df[asset].max()
    print(f"  {asset:15s}: {min_w:7.2%} - {max_w:7.2%}")

print("\n" + "=" * 70)

# Create summary DataFrame
summary_data = {
    'Mean Weight': weights_df.mean(),
    'Min Weight': weights_df.min(),
    'Max Weight': weights_df.max(),
    'Std Dev': weights_df.std(),
    'Latest Weight': latest_weights,
    'Latest Risk Contrib': latest_contrib
}

summary_df = pd.DataFrame(summary_data)
print("\nDetailed Summary Table:")
print(summary_df)

## Extending the Model

### 1. Use Your Own Data

```python
# Load your returns
your_returns = pd.read_csv('your_file.csv', index_col=0, parse_dates=True)

# Run backtest
weights_df, risk_contrib_df, portfolio_vol_df, dates = rolling_rp_backtest(
    returns=your_returns,
    rebalance_freq='M',
    lookback_window=252
)
```

### 2. Try Different Covariance Estimators

Currently uses rolling sample covariance. To implement EWMA or shrinkage:

```python
def estimate_ewma_covariance(returns, lambda_param=0.94):
    """Replace the sample covariance estimator"""
    # Your EWMA implementation
    pass
```

### 3. Modify Optimization

Add constraints like:
- **Maximum position weight**: `bounds = [(0, 0.3) for _ in range(n)]`
- **Minimum position weight**: `bounds = [(0.05, 1.0) for _ in range(n)]`
- **Sector constraints**: Add constraint functions

### 4. Backtest on Real Data

Use actual asset returns to build your Risk Parity strategy on real market data.